# Time Series Activity Forecasting Process

This notebook outlines a detailed process for forecasting activity (e.g., "seen" events) for various indicators over time using a combination of feature engineering, statistical models, machine learning, and ensemble methods. Below is a step-by-step description of the workflow:

---

## 1. Data Loading and Preparation

- **Data Source:** The data is loaded from a CSV file containing time series records for multiple indicators.
- **Preprocessing:**
    - The `date` column is parsed as a datetime object.
    - Data is sorted by `indicator` and `date`.
    - Only the most recent `n_days` (e.g., 100) are retained for analysis.
    - The resulting DataFrame (`df`) contains columns such as `API_UserName`, `date`, `indicator`, `observations`, `dayofweek`, `is_weekend`, `day`, `month`, and `seen`.

---

## 2. Feature Engineering

- **Time Series Features:** For each indicator, the following features are extracted:
    - `last_seen`: Days since the indicator was last seen.
    - `freq_7`, `freq_30`: Number of times seen in the last 7 and 30 days.
    - `avg_gap`: Average gap between seen events.
    - `burstiness`: Variability in the gaps between events.
    - `label_7`, `label_14`, `label_30`: Binary labels indicating if the indicator was seen in the last 7, 14, or 30 days.
- **Output:** A features DataFrame (`features_df`) indexed by indicator.

---

## 3. Model Training and Prediction

- **Models Used:**
    - **Logistic Regression:** Predicts the probability of being seen in the next 7, 14, or 30 days.
    - **Gradient Boosted Trees (GBT):** Another probabilistic classifier for the same task.
    - **Exponential Model:** Uses recent frequency to estimate the probability of being seen.
    - **Weibull AFT Model:** Survival analysis model to estimate the probability of future events.
- **Predictions:** For each indicator, probabilities are computed for being seen today, in 7, 14, and 30 days.

---

## 4. Rule-Based and Ensemble Methods

- **Rule-Based Labels:** Simple heuristics based on `last_seen` (e.g., if last seen within 7 days, label as active).
- **Logistic Model on Rules:** Trains a logistic regression model using the rule-based labels.
- **Ensemble:** Combines probabilities from the logistic model, GBT, Weibull, and exponential models using weighted averages to improve robustness.

---

## 5. Confidence Assessment and Formatting

- **Confidence Labels:** For each forecast window (today, 7d, 14d, 30d), assigns a qualitative confidence label (e.g., "Highly likely", "Possibly active", "Low confidence") based on probability thresholds and recent frequency.
- **Formatting:** Probabilities are formatted as percentages for presentation.

---

## Summary

This process enables robust, interpretable, and actionable forecasting of indicator activity using a blend of statistical and machine learning techniques, with clear outputs for operational use.

In [1]:
import numpy as np
import pandas as pd
import os
import datetime
from datetime import datetime
from datetime import timedelta
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from lifelines import WeibullAFTFitter
import warnings

In [2]:
from datetime import datetime, timedelta

# Set start_full_date_str and start_date_str to 100 days before today
today = datetime.today()
start_full_date_str = (today - timedelta(days=100)).strftime("%Y-%m-%d")
start_date_str = start_full_date_str
# Define the file path template
file_path_template = "Z:/HTOC/Data_Analytics/Data/OpDiv_Observations/htoc_opdiv_obs_d{date}.csv"


In [5]:
# Calculate today's date
today = datetime.today()
print(today)

# Calculate end date (today + 0 days)
end_dt = today + timedelta(days=0)
end_date_str = end_dt.strftime("%Y-%m-%d")
print(end_date_str)

# Convert string dates to datetime objects
start_full_dt = datetime.strptime(start_full_date_str, "%Y-%m-%d")
start_dt = datetime.strptime(start_date_str, "%Y-%m-%d")
# Optionally allow user to set a custom date for 'today'
def set_today(custom_date_str=None):
    global today, end_dt, end_date_str
    if custom_date_str:
        today = datetime.strptime(custom_date_str, "%Y-%m-%d")
        end_dt = today + timedelta(days=0)
        end_date_str = end_dt.strftime("%Y-%m-%d")
        print(f"Custom today set to: {today}")
        print(f"Custom end date: {end_date_str}")

# Example usage:
set_date = set_today("2025-06-16")

2025-07-22 11:51:02.127741
2025-07-22
Custom today set to: 2025-06-16 00:00:00
Custom end date: 2025-06-16


In [6]:
# Function to load files and save them to the specified directory
def load_files(filenames):
    dataframes = []
    for filename in filenames:
        if not os.path.exists(filename):
            print(f"File {filename} does not exist. Skipping.")
            continue
        df = pd.read_csv(filename)
        dataframes.append(df)
    return dataframes


# Define the file path template and date range
date_format = "%Y%m%d"
start_date = datetime.strptime(start_date_str, "%Y-%m-%d")  # Convert start_date_str to datetime
dates_to_pull = pd.date_range(start_date, end_dt, freq='D')

# Generate the list of file paths
datelist = [file_path_template.format(date=dt.strftime(date_format)) for dt in dates_to_pull]
print(datelist)

# Concatenate dataframes from the list of files
src = pd.concat(load_files(datelist), ignore_index=True)

# Clean up the 'indicator' and 'OpDiv' columns if they exist
if 'indicator' in src.columns:
    src['indicator'] = src['indicator'].astype(str).str.split(' ', expand=True)[0].str.strip()
if 'OpDiv' in src.columns:
    src['OpDiv'] = src['OpDiv'].astype(str).str.strip()

# Display the cleaned dataframe
display(src)

['Z:/HTOC/Data_Analytics/Data/OpDiv_Observations/htoc_opdiv_obs_d20250413.csv', 'Z:/HTOC/Data_Analytics/Data/OpDiv_Observations/htoc_opdiv_obs_d20250414.csv', 'Z:/HTOC/Data_Analytics/Data/OpDiv_Observations/htoc_opdiv_obs_d20250415.csv', 'Z:/HTOC/Data_Analytics/Data/OpDiv_Observations/htoc_opdiv_obs_d20250416.csv', 'Z:/HTOC/Data_Analytics/Data/OpDiv_Observations/htoc_opdiv_obs_d20250417.csv', 'Z:/HTOC/Data_Analytics/Data/OpDiv_Observations/htoc_opdiv_obs_d20250418.csv', 'Z:/HTOC/Data_Analytics/Data/OpDiv_Observations/htoc_opdiv_obs_d20250419.csv', 'Z:/HTOC/Data_Analytics/Data/OpDiv_Observations/htoc_opdiv_obs_d20250420.csv', 'Z:/HTOC/Data_Analytics/Data/OpDiv_Observations/htoc_opdiv_obs_d20250421.csv', 'Z:/HTOC/Data_Analytics/Data/OpDiv_Observations/htoc_opdiv_obs_d20250422.csv', 'Z:/HTOC/Data_Analytics/Data/OpDiv_Observations/htoc_opdiv_obs_d20250423.csv', 'Z:/HTOC/Data_Analytics/Data/OpDiv_Observations/htoc_opdiv_obs_d20250424.csv', 'Z:/HTOC/Data_Analytics/Data/OpDiv_Observations/hto

,indicator,API_UserName,obs_date,OpDiv,indicator_key,observations,curr_date
0,103.140.63.154,15920309593055310684,2025-04-13,CMS,103.140.63.154_CMS,2517,2025-04-13
1,104.131.6.219,20790633968691748718,2025-04-13,DHA,104.131.6.219_DHA,2,2025-04-13
2,104.160.6.2,20790633968691748718,2025-04-13,DHA,104.160.6.2_DHA,2,2025-04-13
3,104.160.6.2,80363974983666420473,2025-04-13,IHS,104.160.6.2_IHS,2,2025-04-13
4,104.18.32.191,80363974983666420473,2025-04-13,IHS,104.18.32.191_IHS,5,2025-04-13
...,...,...,...,...,...,...,...
56173,www.shorturl.at/,13983566077459384977,2025-06-16,FDA,www.shorturl.at/_FDA,4,2025-06-16
56174,www.sthda.com,15920309593055310684,2025-06-16,CMS,www.sthda.com_CMS,4,2025-06-16
56175,www.sthda.com,20790633968691748718,2025-06-16,DHA,www.sthda.com_DHA,117,2025-06-16
56176,www.sthda.com,50189120947314147395,2025-06-16,NIH,www.sthda.com_NIH,68,2025-06-16


In [5]:
src.drop(columns=['curr_date', 'indicator_key'], inplace=True)
src.rename(columns={'obs_date': 'date'}, inplace=True)
src['date'] = pd.to_datetime(src['date'])
src.reset_index(drop=True, inplace=True)


In [8]:
# Group the src dataframe by 'OpDiv' and get a dictionary of DataFrames for each OpDiv
opdiv_groups = {opdiv: group for opdiv, group in src.groupby('OpDiv')}

# Allow searching by indicator
def get_by_indicator(indicator_value):
    return src[src['indicator'] == indicator_value]

# Example usage:
get_by_indicator('192.124.249.112')

,indicator,API_UserName,obs_date,OpDiv,indicator_key,observations,curr_date
1068,192.124.249.112,50189120947314147395,2025-04-15,NIH,192.124.249.112_NIH,4,2025-04-15
1757,192.124.249.112,80363974983666420473,2025-04-16,IHS,192.124.249.112_IHS,13,2025-04-16
1758,192.124.249.112,50189120947314147395,2025-04-16,NIH,192.124.249.112_NIH,151,2025-04-16
1759,192.124.249.112,74198686107399967946,2025-04-16,VA,192.124.249.112_VA,14,2025-04-16
2496,192.124.249.112,00818860012482918321,2025-04-17,CDC,192.124.249.112_CDC,7,2025-04-17
...,...,...,...,...,...,...,...
53493,192.124.249.112,74198686107399967946,2025-06-15,VA,192.124.249.112_VA,4,2025-06-15
54997,192.124.249.112,00818860012482918321,2025-06-16,CDC,192.124.249.112_CDC,91,2025-06-16
54998,192.124.249.112,80363974983666420473,2025-06-16,IHS,192.124.249.112_IHS,8,2025-06-16
54999,192.124.249.112,50189120947314147395,2025-06-16,NIH,192.124.249.112_NIH,52,2025-06-16


In [13]:
opdiv_merged = {}

for opdiv, group_df in opdiv_groups.items():
    group_df['date'] = pd.to_datetime(group_df['obs_date'])
    all_users = group_df['API_UserName'].unique()
    all_indicators = group_df['indicator'].unique()
    all_dates = pd.date_range(start=group_df['date'].min(), end=pd.Timestamp.now().normalize(), freq='D')
    all_combinations = pd.MultiIndex.from_product(
        [all_users, all_dates, all_indicators],
        names=['API_UserName', 'date', 'indicator']
    ).to_frame(index=False)
    all_combinations['OpDiv'] = opdiv  # Add OpDiv column

    merged = all_combinations.merge(group_df, how='left', on=['API_UserName', 'date', 'indicator', 'OpDiv'])
    merged['observations'] = merged['observations'].fillna(0).astype(int)
    merged['date'] = pd.to_datetime(merged['date'])
    merged['dayofweek'] = merged['date'].dt.dayofweek
    merged['is_weekend'] = merged['dayofweek'].isin([5, 6])
    merged['day'] = merged['date'].dt.day
    merged['month'] = merged['date'].dt.month
    merged['seen'] = (merged['observations'] > 0).astype(int)
    # Add total number of seen for each indicator
    seen_totals = merged.groupby('indicator')['seen'].sum().rename('total_seen')
    merged = merged.merge(seen_totals, on='indicator', how='left')
    # Filter out indicators seen only 1 time
    merged = merged[merged['total_seen'] > 1].reset_index(drop=True)
    opdiv_merged[opdiv] = merged


# Example: display the merged dataframe for one OpDiv
display(opdiv_merged['DHA'])


,API_UserName,date,indicator,OpDiv,obs_date,indicator_key,observations,curr_date,dayofweek,is_weekend,day,month,seen,total_seen
0,20790633968691748718,2025-04-13,104.131.6.219,DHA,2025-04-13,104.131.6.219_DHA,2,2025-04-13,6,True,13,4,1,13
1,20790633968691748718,2025-04-13,104.160.6.2,DHA,2025-04-13,104.160.6.2_DHA,2,2025-04-13,6,True,13,4,1,62
2,20790633968691748718,2025-04-13,118.193.59.10,DHA,2025-04-13,118.193.59.10_DHA,2,2025-04-13,6,True,13,4,1,41
3,20790633968691748718,2025-04-13,118.193.72.187,DHA,2025-04-13,118.193.72.187_DHA,6,2025-04-13,6,True,13,4,1,35
4,20790633968691748718,2025-04-13,146.71.50.198,DHA,2025-04-13,146.71.50.198_DHA,340975,2025-04-13,6,True,13,4,1,17
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
52818,20790633968691748718,2025-07-22,138.199.21.227,DHA,NaN,NaN,0,NaN,1,False,22,7,0,2
52819,20790633968691748718,2025-07-22,149.88.22.169,DHA,NaN,NaN,0,NaN,1,False,22,7,0,3
52820,20790633968691748718,2025-07-22,169.150.201.3,DHA,NaN,NaN,0,NaN,1,False,22,7,0,3
52821,20790633968691748718,2025-07-22,178.62.1.211,DHA,NaN,NaN,0,NaN,1,False,22,7,0,3


In [14]:
# Replace 'VA' with the correct OpDiv if needed
opdiv_merged['VA'][opdiv_merged['VA']['indicator'] == '192.124.249.112']

,API_UserName,date,indicator,OpDiv,obs_date,indicator_key,observations,curr_date,dayofweek,is_weekend,day,month,seen,total_seen
83,74198686107399967946,2025-04-13,192.124.249.112,VA,NaN,NaN,0,NaN,6,True,13,4,0,62
603,74198686107399967946,2025-04-14,192.124.249.112,VA,NaN,NaN,0,NaN,0,False,14,4,0,62
1123,74198686107399967946,2025-04-15,192.124.249.112,VA,NaN,NaN,0,NaN,1,False,15,4,0,62
1643,74198686107399967946,2025-04-16,192.124.249.112,VA,2025-04-16,192.124.249.112_VA,14,2025-04-16,2,False,16,4,1,62
2163,74198686107399967946,2025-04-17,192.124.249.112,VA,2025-04-17,192.124.249.112_VA,35,2025-04-17,3,False,17,4,1,62
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
50003,74198686107399967946,2025-07-18,192.124.249.112,VA,NaN,NaN,0,NaN,4,False,18,7,0,62
50523,74198686107399967946,2025-07-19,192.124.249.112,VA,NaN,NaN,0,NaN,5,True,19,7,0,62
51043,74198686107399967946,2025-07-20,192.124.249.112,VA,NaN,NaN,0,NaN,6,True,20,7,0,62
51563,74198686107399967946,2025-07-21,192.124.249.112,VA,NaN,NaN,0,NaN,0,False,21,7,0,62


In [15]:


def load_data(filepath, n_days=100):
    df = pd.read_csv(filepath)
    df['date'] = pd.to_datetime(df['date'])
    df.sort_values(by=['indicator', 'date'], inplace=True)
    latest_dates = df['date'].drop_duplicates().sort_values().tail(n_days)
    return df[df['date'].isin(latest_dates)].copy()

def extract_time_series_features(group):
    series = group['seen'].values
    indices = np.where(series == 1)[0]
    if len(indices) == 0:
        return pd.Series({
            'last_seen': len(series),
            'freq_7': 0,
            'freq_14': 0,
            'freq_30': 0,
            'avg_gap': len(series),
            'burstiness': 0,
            'label_7': 0,
            'label_14': 0,
            'label_30': 0
        })
    last_seen = len(series) - 1 - indices[-1]
    freq_7 = np.sum(series[-7:])
    freq_14 = np.sum(series[-14:])
    freq_30 = np.sum(series[-30:])
    gaps = np.diff(indices)
    avg_gap = np.mean(gaps) if len(gaps) > 0 else len(series)
    burstiness = (np.std(gaps) - avg_gap) / (np.std(gaps) + avg_gap) if len(gaps) > 1 else 0
    label_7 = 1 if np.any(series[-7:]) else 0
    label_14 = 1 if np.any(series[-14:]) else 0
    label_30 = 1 if np.any(series[-30:]) else 0
    return pd.Series({
        'last_seen': last_seen,
        'freq_7': freq_7,
        'freq_14': freq_14,
        'freq_30': freq_30,
        'avg_gap': avg_gap,
        'burstiness': burstiness,
        'label_7': label_7,
        'label_14': label_14,
        'label_30': label_30
    })

def build_features(df):
    features_df = df.groupby('indicator').apply(extract_time_series_features).reset_index()
    return features_df

def train_predict(model_cls, X, y):
    if len(np.unique(y)) < 2:
        return np.full(len(y), np.nan)
    model = Pipeline([('scaler', StandardScaler()), ('clf', model_cls())])
    model.fit(X, y)
    return model.predict_proba(X)[:, 1]

def train_gbt(X, y):
    if len(np.unique(y)) < 2:
        return np.full(len(y), np.nan)
    model = GradientBoostingClassifier()
    model.fit(X, y)
    return model.predict_proba(X)[:, 1]

def fit_weibull_aft(X, avg_gap, event):
    aft_df = X.copy()
    aft_df['duration'] = avg_gap
    aft_df['event'] = event
    aft = WeibullAFTFitter()
    aft.fit(aft_df, duration_col='duration', event_col='event')
    return aft

def get_model_outputs(features_df, df):
    df_pred = features_df.copy()
    X = df_pred[['last_seen', 'freq_7', 'freq_14' ,'freq_30', 'avg_gap', 'burstiness']]
    y_7 = df_pred['label_7']
    y_14 = df_pred['label_14']
    y_30 = df_pred['label_30']

    # Logistic Regression, Estimate baseline probabilities
    df_pred['logistic_7'] = train_predict(LogisticRegression, X, y_7)
    df_pred['logistic_14'] = train_predict(LogisticRegression, X, y_14)
    df_pred['logistic_30'] = train_predict(LogisticRegression, X, y_30)

    # Gradient Boosted Trees, Non-linear patterns and feature interactions
    df_pred['gbt_7'] = train_gbt(X, y_7)
    df_pred['gbt_14'] = train_gbt(X, y_14)
    df_pred['gbt_30'] = train_gbt(X, y_30)

    # Exponential Model, memoryless Poisson process for frequency
    rate = (df_pred['freq_30'] / 30).clip(lower=1e-6)
    df_pred['exp_7'] = 1 - np.exp(-rate * 7)
    df_pred['exp_14'] = 1 - np.exp(-rate * 14)
    df_pred['exp_30'] = 1 - np.exp(-rate * 30)
    df_pred['exp_today'] = 1 - np.exp(-rate * 1)

    # Weibull AFT Model, time-to-event behavor, burstiness
    aft = fit_weibull_aft(X, df_pred['avg_gap'], y_7)
    surv_func = aft.predict_survival_function(X.assign(duration=df_pred['avg_gap'], event=y_7), times=[1, 7, 14, 30])
    df_pred['weibull_today'] = 1 - surv_func.loc[1].values
    df_pred['weibull_7'] = 1 - surv_func.loc[7].values
    df_pred['weibull_14'] = 1 - surv_func.loc[14].values
    df_pred['weibull_30'] = 1 - surv_func.loc[30].values

    # Today's forecast
    df_pred['logistic_today'] = train_predict(LogisticRegression, X, y_7)
    df_pred['gbt_today'] = train_gbt(X, y_7)

    # Merge in actual "seen" value for today's date
    latest_date = df['date'].max()
    today_seen = df[df['date'] == latest_date][['indicator', 'seen']].rename(columns={'seen': 'seen_today'})
    df_pred = df_pred.merge(today_seen, on='indicator', how='left')

    return df_pred

def add_rule_and_ensemble(output):
    features = ['last_seen', 'freq_7', 'freq_30', 'avg_gap', 'burstiness']
    X = output[features]

    # Rule-based labels
    output['rule_today'] = output['last_seen'].apply(lambda x: 1 if x == 0 else 0)
    output['rule_7d'] = output['last_seen'].apply(lambda x: 1 if x <= 6 else 0)
    output['rule_14d'] = output['last_seen'].apply(lambda x: 1 if x <= 13 else 0)
    output['rule_30d'] = output['last_seen'].apply(lambda x: 1 if x <= 29 else 0)

    y_today = output['rule_today']
    y_7 = output['rule_7d']
    y_14 = output['rule_14d']
    y_30 = output['rule_30d']

    def train_logistic_model(X, y):
        if len(np.unique(y)) < 2:
            return np.full(len(y), np.nan)
        model = Pipeline([
            ('scaler', StandardScaler()),
            ('clf', LogisticRegression())
        ])
        model.fit(X, y)
        return model.predict_proba(X)[:, 1]

    output['prob_today'] = train_logistic_model(X, y_today)
    output['prob_7d'] = train_logistic_model(X, y_7)
    output['prob_14d'] = train_logistic_model(X, y_14)
    output['prob_30d'] = train_logistic_model(X, y_30)

    # Ensemble, combines predictions from all models and prevents overfitting
    output['ensemble_7d'] = (
        0.3 * output['prob_7d'].astype(float) +
        0.25 * output['gbt_7'] +
        0.25 * output['weibull_7'] +
        0.3 * output['exp_7']
    )
    output['ensemble_14d'] = (
        0.3 * output['prob_14d'].astype(float) +
        0.25 * output['gbt_14'] +
        0.25 * output['weibull_14'] +
        0.3 * output['exp_14']
    )
    output['ensemble_30d'] = (
        0.3 * output['prob_30d'].astype(float) +
        0.25 * output['gbt_30'] +
        0.25 * output['weibull_30'] +
        0.3 * output['exp_30']
    )
    return output

def classify_window(prob, freq, high_thresh, label):
    if prob >= high_thresh and freq >= 2:
        return f"{label}: Highly likely"
    elif prob >= 0.07 and freq >= 1:
        return f"{label}: Possibly active"
    else:
        return f"{label}: Low confidence"

def add_confidence_and_format(output):
    output['confidence_7d'] = output.apply(
        lambda row: classify_window(float(row['ensemble_7d']), row['freq_7'], 0.6, '7-Day'), axis=1
    )
    output['confidence_14d'] = output.apply(
        lambda row: classify_window(float(row['ensemble_14d']), row['freq_14'], 0.6, '14-Day'), axis=1
    )
    output['confidence_30d'] = output.apply(
        lambda row: classify_window(float(row['ensemble_30d']), row['freq_30'], 0.6, '30-Day'), axis=1
    )

    # Format percentages
    for col in ['prob_7d', 'prob_14d', 'prob_30d', 'ensemble_7d', 'ensemble_14d', 'ensemble_30d']:
        output[col] = np.clip(output[col].astype(float) * 100, 0, 100).round(2).astype(str) + '%'
    return output

def build_production_output(output):
    warnings.simplefilter(action='ignore', category=pd.errors.SettingWithCopyWarning)
    production_output = output[[
        'indicator', 'seen_today', 'freq_7', 'freq_30',
        'ensemble_7d', 'confidence_7d',
        'ensemble_14d', 'confidence_14d',
        'ensemble_30d', 'confidence_30d'
    ]].copy()
    production_output.rename(columns={
        'indicator': 'Indicator',
        'seen_today': 'Observed Today',
        'freq_7': 'Frequency (7d)',
        'freq_30': 'Frequency (30d)',
        'ensemble_7d': 'Probability: 7-Day',
        'confidence_7d': 'Confidence: 7-Day',
        'ensemble_14d': 'Probability: 14-Day',
        'confidence_14d': 'Confidence: 14-Day',
        'ensemble_30d': 'Probability: 30-Day',
        'confidence_30d': 'Confidence: 30-Day'
    }, inplace=True)
    return production_output


In [16]:
import warnings

# Silence pandas groupby apply deprecation warning globally for this notebook
warnings.filterwarnings("ignore", category=DeprecationWarning, message="DataFrameGroupBy.apply operated on the grouping columns")

from datetime import datetime
import os

def main():
    today = datetime.today().date()

    prediction_path = r'C:\Users\jaskew\Documents\NOI Logs\Predictions'

    opdiv_production_outputs = {}
    opdiv_forecast_logs = {}
    production_output = None
    opdiv_df = None

    for opdiv_name, opdiv_df in opdiv_merged.items():
        opdiv_df = opdiv_df.copy()
        features_df = build_features(opdiv_df)
        output = get_model_outputs(features_df, opdiv_df)
        output = add_rule_and_ensemble(output)
        output = add_confidence_and_format(output)
        production_output = build_production_output(output)


        # Save production output
        opdiv_production_outputs[opdiv_name] = production_output

        # Save today's prediction CSV
        opdiv_output_dir = os.path.join(prediction_path, opdiv_name)
        os.makedirs(opdiv_output_dir, exist_ok=True)

    # Explicit unpacking of key OpDivs (optional for clarity)
    DHA_output = opdiv_production_outputs.get("DHA")
    CDC_output = opdiv_production_outputs.get("CDC")
    FDA_output = opdiv_production_outputs.get("FDA")
    NIH_output = opdiv_production_outputs.get("NIH")
    VA_output = opdiv_production_outputs.get("VA")
    HRSA_output = opdiv_production_outputs.get("HRSA")
    IHS_output = opdiv_production_outputs.get("IHS")
    OS_output = opdiv_production_outputs.get("OS")
    CMS_output = opdiv_production_outputs.get("CMS")
    HHS_output = opdiv_production_outputs.get("HHS")

    DHA_log = opdiv_forecast_logs.get("DHA")
    CDC_log = opdiv_forecast_logs.get("CDC")
    FDA_log = opdiv_forecast_logs.get("FDA")
    NIH_log = opdiv_forecast_logs.get("NIH")
    VA_log = opdiv_forecast_logs.get("VA")
    HRSA_log = opdiv_forecast_logs.get("HRSA")
    IHS_log = opdiv_forecast_logs.get("IHS")
    OS_log = opdiv_forecast_logs.get("OS")
    CMS_log = opdiv_forecast_logs.get("CMS")
    HHS_log = opdiv_forecast_logs.get("HHS")

    display(production_output.head(5))
    display(opdiv_df.head(5))

    return {
        "DHA_output": DHA_output, "CDC_output": CDC_output, "FDA_output": FDA_output,
        "NIH_output": NIH_output, "VA_output": VA_output, "HRSA_output": HRSA_output,
        "IHS_output": IHS_output, "OS_output": OS_output, "CMS_output": CMS_output,
        "HHS_output": HHS_output,
        "DHA_log": DHA_log, "CDC_log": CDC_log, "FDA_log": FDA_log,
        "NIH_log": NIH_log, "VA_log": VA_log, "HRSA_log": HRSA_log,
        "IHS_log": IHS_log, "OS_log": OS_log, "CMS_log": CMS_log,
        "HHS_log": HHS_log
    }


if __name__ == "__main__":
    main()

C:\Users\jaskew\AppData\Local\Temp\ipykernel_16884\3775426915.py:46: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  features_df = df.groupby('indicator').apply(extract_time_series_features).reset_index()
C:\Users\jaskew\AppData\Roaming\Python\Python313\site-packages\lifelines\fitters\__init__.py:2086: ApproximationWarning: The Hessian was not invertible. We will instead approximate it using the pseudo-inverse.

It's advisable to not trust the variances reported, and to be suspicious of the fitted parameters too.

Some ways to possible ways fix this:

0. Are there any lifelines warnings outputted during the `fit`?
1. Does a particularly large variable need to be centered to 0?
2. Inspect your DataFrame:

,Indicator,Observed Today,Frequency (7d),Frequency (30d),Probability: 7-Day,Confidence: 7-Day,Probability: 14-Day,Confidence: 14-Day,Probability: 30-Day,Confidence: 30-Day
0,103.133.107.28,0,0.0,0.0,nan%,7-Day: Low confidence,nan%,14-Day: Low confidence,nan%,30-Day: Low confidence
1,103.147.185.248,0,0.0,0.0,nan%,7-Day: Low confidence,nan%,14-Day: Low confidence,nan%,30-Day: Low confidence
2,103.149.249.226,0,0.0,0.0,nan%,7-Day: Low confidence,nan%,14-Day: Low confidence,nan%,30-Day: Low confidence
3,103.149.86.208,0,0.0,0.0,nan%,7-Day: Low confidence,nan%,14-Day: Low confidence,nan%,30-Day: Low confidence
4,103.153.78.59,0,0.0,0.0,nan%,7-Day: Low confidence,nan%,14-Day: Low confidence,nan%,30-Day: Low confidence


,API_UserName,date,indicator,OpDiv,obs_date,indicator_key,observations,curr_date,dayofweek,is_weekend,day,month,seen,total_seen
0,74198686107399967946,2025-04-13,104.21.3.76,VA,2025-04-13,104.21.3.76_VA,2,2025-04-13,6,True,13,4,1,42
1,74198686107399967946,2025-04-13,118.193.59.10,VA,2025-04-13,118.193.59.10_VA,19,2025-04-13,6,True,13,4,1,52
2,74198686107399967946,2025-04-13,118.193.72.187,VA,2025-04-13,118.193.72.187_VA,36,2025-04-13,6,True,13,4,1,53
3,74198686107399967946,2025-04-13,146.70.45.166,VA,2025-04-13,146.70.45.166_VA,7,2025-04-13,6,True,13,4,1,65
4,74198686107399967946,2025-04-13,149.88.27.212,VA,2025-04-13,149.88.27.212_VA,1,2025-04-13,6,True,13,4,1,22


In [ ]:
# Function to set a custom prediction date
def set_prediction_date(custom_date_str):
    global today, end_dt, end_date_str
    today = datetime.strptime(custom_date_str, "%Y-%m-%d").date()
    end_dt = today + timedelta(days=0)
    end_date_str = end_dt.strftime("%Y-%m-%d")
    print(f"Prediction date set to: {today}")
    print(f"End date: {end_date_str}")

# Example usage: Set a custom prediction date
set_prediction_date("2025-06-16")